# RDF export of CMDI vocabularies and VLO facets in relation to concepts
Note: this notebook depends on the output of the [clarin_data](clarin_data.ipynb) notebook. The output should already be present, but if you run into problems please run this first to make sure that the data is complete and up-to-date.

In [1]:
# Preamble

# Define some common constants
CCR_API_BASE = 'https://vocabularies.clarin.eu/clavas/rest/v1/'
CCR_VOCAB = 'ccr'
VLO_API_BASE = 'https://vlo.clarin-dev.eu/api'
COMPONENT_REGISTRY_BASE = 'https://catalog.clarin.eu/ds/ComponentRegistry/rest'

# Directories from data collection notebook
DATA_DIR = 'data'
CCR_DATA_DIR = DATA_DIR + '/ccr'
PROFILES_DATA_DIR = DATA_DIR + '/profiles'
VLO_FACETS_DATA_DIR = DATA_DIR + '/facets'
VLO_MAPPING_DATA_DIR = DATA_DIR + '/mappings'

import os
# Create output directories for data
RDF_DIR = 'rdf'
CCR_RDF_DIR = RDF_DIR + '/ccr'
COMPONENT_REGISTRY_RDF_DIR = RDF_DIR + '/componentRegistry'
FACETS_RDF_DIR = RDF_DIR + '/facets'
for dataDir in [RDF_DIR, CCR_RDF_DIR, COMPONENT_REGISTRY_RDF_DIR, FACETS_RDF_DIR]:
    if not os.path.exists(dataDir):
        print("Creating data directory: ", dataDir)
        os.mkdir(dataDir)

## Concept definitions form the CLARIN Concept Registry
The output of [clarin_data](clarin_data.ipynb) contains individual RDF/XML exports of the concept schemes of the CCR. Here where load these into a graph and export them into a single Turtle file.

In [2]:
# Read concepts and save as Turtle
from rdflib import Graph
import pprint

conceptsGraph = Graph()
for root, dirs, files in os.walk(CCR_DATA_DIR):
    for file in files:
        print('Reading RDF XML:', file)
        conceptsGraph.parse(CCR_DATA_DIR + '/' + file)
        # TODO: serialzie as Turtle in CCR_RDF_DIR

Reading RDF XML: CCR_P-LanguageResourceOntology_05399e1d-4f23-8fc1-5088-eb5afbf0cd91.xml
Reading RDF XML: CCR_P-Morphosyntax_f701a84e-9171-1e2d-22cf-6528c2571d42.xml
Reading RDF XML: CCR_P-notavailable_dc55639d-6b2b-a05a-ced0-9d394432ae43.xml
Reading RDF XML: CCR_P-Metadata_6f3f84d1-6f06-6291-4e20-4cd361cca128.xml
Reading RDF XML: CCR_P-Terminology_5b77056e-4634-1b16-a143-3697c33c17a7.xml
Reading RDF XML: CCR_P-SemanticContentRepresentation_1448f330-771b-ad11-987f-7293629f0801.xml
Reading RDF XML: CCR_P-LexicalResources_ce3edd5c-07a7-b345-dcfa-789a9b4bc980.xml
Reading RDF XML: CCR_P-SignLanguage_8f51ce1b-211d-9682-a8e7-aeb0f2a79e03.xml
Reading RDF XML: CCR_P-DialogueActs_955814a6-6c07-c143-94eb-b2551c2d51cb.xml
Reading RDF XML: CCR_P-LanguageCodes_a122c1a3-1912-fecd-07a2-8685522dfeca.xml
Reading RDF XML: CCR_P-Syntax_e7dbb854-a006-9939-4863-1236a4062a1b.xml
Reading RDF XML: CCR_P-undecided_0103ef85-e040-85f0-c3a1-7b26f5659821.xml
Reading RDF XML: CCR_P-Translation_e7ebeaa2-68f1-34c8-63

In [3]:
ccrRdfOutFile = CCR_RDF_DIR + '/concepts.ttl'

print('Serializing combined graph...')
conceptsGraph.serialize(destination = ccrRdfOutFile)
print('Combined graph written to', ccrRdfOutFile)


Serializing combined graph...
Combined graph written to rdf/ccr/concepts.ttl


## Controlled vocabularies from CMDI component/profile specifications
An XSLT based solution to generate RDF statements for all controlled vocabularies and their items defined in the ComponentRegistry is available at [https://github.com/clarin-eric/metacat-cmd](https://github.com/clarin-eric/metacat-cmd). In the first cell below we simply parse the output of a run on all used profiles (the output of [clarin_data](clarin_data.ipynb)). 

At a later stage we may integrate/replicate the logic in this notebook.

In [4]:
### Load from pregenerated export
compRegInFile = COMPONENT_REGISTRY_RDF_DIR + '/clarin-all.ttl'
compRegOutFile = COMPONENT_REGISTRY_RDF_DIR + '/compreg-vocabs.ttl'

if not os.path.exists(compRegInFile):
    raise Exception('Input file ' + compRegInFile + 'not found. Please put this file in place to allow for processing.')

compRegGraph = Graph()
print('Parsing Component Registry vocabularies')
compRegGraph.parse(compRegInFile)

print('Serializing CompReg vocabularies graph...')
compRegGraph.serialize(destination = compRegOutFile)
print('CompReg vocabularies graph written to', compRegOutFile)

Parsing Component Registry vocabularies
Serializing CompReg vocabularies graph...
CompReg vocabularies graph written to rdf/componentRegistry/compreg-vocabs.ttl


In [5]:
#### TODO: Produce on the spot
# Loop over data/profiles/*.xml
# Apply XSLT
# Load and store results in a single ttl file

## VLO facet definitions
For the VLO facets, we will produce two sets of statements: one for the facet-concept mapping (extracting information from [facetConcepts.xml](data/mappings/facetConcepts.xml); and another for the facet values which were gathered from the VLO API and stored in [vlo-facets.json](data/facets/vlo-facets.json).

In [6]:
from rdflib import URIRef, Namespace
VLO = Namespace('https://vlo.clarin.eu/')
VLO_FACET = Namespace('https://vlo.clarin.eu/facets#')
CCR = Namespace('http://hdl.handle.net/11459/')
ISOCAT = Namespace('http://www.isocat.org/datcat/')

In [9]:
### Facet - concept mapping
facetConceptMappingInFile = VLO_MAPPING_DATA_DIR + '/facetConcepts.xml'
facetConceptMappingOutFile = FACETS_RDF_DIR + '/facets.ttl'

# Load the mapping file
from lxml import etree
xmlTree = etree.parse(facetConceptMappingInFile)
root = xmlTree.getroot()

facets = root.findall('facetConcept')
print('Found mapping definitions for', len(facets), 'facets')

# Construct the graph
from rdflib.namespace import RDF, DC
facetsGraph = Graph()
facetsGraph.bind('vlo',VLO)
facetsGraph.bind('facet',VLO_FACET)
facetsGraph.bind('ccr', CCR)
facetsGraph.bind('isocat',ISOCAT)

for facet in facets: #TODO: filter out fields that are not facets
    facetName = facet.attrib['name']
    if facetName:
        facetRef = URIRef('https://vlo.clarin.eu/facets#' + facetName)
        facetsGraph.add((facetRef, RDF.type, VLO.Facet))
        
        # Traverse concept mappings for this facet
        concepts = facet.findall('concept')
        print('Facet:', facetName, '-', len(concepts), 'concepts')
        for concept in concepts:
            if concept.text:
                conceptRef = URIRef(concept.text)
                facetsGraph.add((facetRef, VLO.facetConceptLink, conceptRef))
            
print('Serializing facets graph...')
facetsGraph.serialize(destination = facetConceptMappingOutFile)
print('Facets graph written to', facetConceptMappingOutFile)

Found mapping definitions for 28 facets
Facet: id - 2 concepts
Facet: _selfLink - 0 concepts
Facet: _normalizedSelfLink - 0 concepts
Facet: collection - 0 concepts
Facet: projectName - 6 concepts
Facet: name - 20 concepts
Facet: temporalCoverage - 19 concepts
Facet: continent - 4 concepts
Facet: country - 6 concepts
Facet: languageCode - 11 concepts
Facet: _languageName - 0 concepts
Facet: availability - 16 concepts
Facet: licenseType - 1 concepts
Facet: license - 13 concepts
Facet: accessInfo - 8 concepts
Facet: organisation - 8 concepts
Facet: genre - 4 concepts
Facet: modality - 2 concepts
Facet: subject - 14 concepts
Facet: description - 6 concepts
Facet: resourceClass - 9 concepts
Facet: format - 3 concepts
Facet: text - 0 concepts
Facet: keywords - 4 concepts
Facet: lifeCycleStatus - 2 concepts
Facet: distributionType - 2 concepts
Facet: rightsHolder - 4 concepts
Facet: creator - 9 concepts
Serializing facets graph...
Facets graph written to rdf/facets/facets.ttl
